In [1]:
import sys
sys.path.insert(0, '..')

In [2]:
from premia_junior.models.black_scholes import BlackScholes
from premia_junior.methods.analytical import Analytical
from premia_junior.products.european import EuropeanCall
from premia_junior.engine.pricer import Pricer

model = BlackScholes(mu=0.05, sigma=0.2)
method = Analytical()
product = EuropeanCall(K=100)

pricer = Pricer(model=model, method=method, product=product)
result = pricer.price(S0=100, T=1.0, r=0.05)
print(result)

PricingResult(price=10.450583572185565, confidence_interval=None, metadata={'method': 'Analytical', 'model': 'BlackScholes', 'd1': 0.35000000000000003, 'd2': 0.15000000000000002})


In [3]:
from premia_junior.methods.monte_carlo import MonteCarlo

mc = MonteCarlo(n_paths=100000, n_steps=252, seed=42)
pricer_mc = Pricer(model=model, method=mc, product=product)
result_mc = pricer_mc.price(S0=100, T=1.0, r=0.05)
print(result_mc)

PricingResult(price=10.482858080322606, confidence_interval=(10.390945307384213, 10.574770853260997), metadata={'Method': 'MonteCarlo', 'Model': 'BlackScholes', 'Product': 'EuropeanCall'})


In [4]:
import numpy as np
from premia_junior.models.black_scholes import BlackScholes
from premia_junior.methods.CRR import CRR
from premia_junior.methods.analytical import Analytical
from premia_junior.products.european import EuropeanPut
from premia_junior.products.american import AmericanPut
from premia_junior.engine.pricer import Pricer

model = BlackScholes(mu=0.05, sigma=0.2)
S0, K, T, r = 100, 100, 1.0, 0.05

# European put — CRR vs Analytical
eu_put = EuropeanPut(K=K)
crr = CRR(n_steps=500)

r1 = Pricer(model=model, method=Analytical(), product=eu_put).price(S0, T, r)
r2 = Pricer(model=model, method=crr, product=eu_put).price(S0, T, r)
print(f"European put — Analytical: {r1.price:.4f}, CRR: {r2.price:.4f}")

# American put — CRR
am_put = AmericanPut(K=K)
r3 = Pricer(model=model, method=crr, product=am_put).price(S0, T, r)
print(f"American put — CRR: {r3.price:.4f}")

European put — Analytical: 5.5735, CRR: 5.5695
American put — CRR: 6.0888


In [5]:
from premia_junior.models.heston import Heston
from premia_junior.methods.monte_carlo import MonteCarlo
from premia_junior.products.european import EuropeanCall
from premia_junior.engine.pricer import Pricer

model = Heston(v0=0.04, kappa=1.5, theta=0.04, xi=0.3, rho=-0.7)
method = MonteCarlo(n_paths=100000, n_steps=252)
product = EuropeanCall(K=100)

result = Pricer(model=model, method=method, product=product).price(S0=100, T=1.0, r=0.05)
print(result)

Feller Condition is verified So the variance is always positive
PricingResult(price=10.367484573194185, confidence_interval=(10.29319246751938, 10.441776678868989), metadata={'Method': 'MonteCarlo', 'Model': 'Heston', 'Product': 'EuropeanCall'})


In [6]:
from premia_junior.models.black_scholes import BlackScholes
from premia_junior.methods.monte_carlo import MonteCarlo
from premia_junior.products.asian import DiscreteAsianCall
from premia_junior.engine.pricer import Pricer

model = BlackScholes(mu=0.05, sigma=0.2)
method = MonteCarlo(n_paths=100000, n_steps=252)
product = DiscreteAsianCall(K=100)

result = Pricer(model=model, method=method, product=product).price(S0=100, T=1.0, r=0.05)
print(result)


PricingResult(price=5.770525303348857, confidence_interval=(5.7210690581713415, 5.819981548526373), metadata={'Method': 'MonteCarlo', 'Model': 'BlackScholes', 'Product': 'DiscreteAsianCall'})


In [7]:
import sys
sys.path.insert(0, '..')

from premia_junior.models.black_scholes import BlackScholes
from premia_junior.methods.analytical import Analytical
from premia_junior.products.european import EuropeanCall
from premia_junior.engine.pricer import Pricer
from premia_junior.utils.greek import FiniteDifference
from premia_junior.utils.BaseGreekScheme import CentralDifference1

model   = BlackScholes(mu=0.05, sigma=0.2)
pricer  = Pricer(model=model, method=Analytical(), product=EuropeanCall(K=100))
fd      = FiniteDifference(CentralDifference1(), h=0.01)

# 1. with_params: the new model changes, the original doesn't
print(model.with_params(sigma=0.21).sigma, model.sigma)   # 0.21  0.2

# 2. the five Greeks
S0, T, r = 100, 1.0, 0.05
print("delta", fd.delta(pricer, S0, T, r))
print("gamma", fd.gamma(pricer, S0, T, r))
print("vega ", fd.vega(pricer, S0, T, r))
print("rho  ", fd.rho(pricer, S0, T, r))
print("theta", fd.theta(pricer, S0, T, r))

0.21 0.2
delta (0.6368306425763137, None)
gamma (0.018762017077733617, None)
vega  (37.52098305590863, None)
rho   (0.0, None)
theta (0.03231170209474499, None)


In [8]:
from premia_junior.methods.monte_carlo import MonteCarlo

S0, T, r = 100, 1.0, 0.05
product = EuropeanCall(K=100)

mc_crn   = MonteCarlo(n_paths=100000, n_steps=252, seed=42)    # same random numbers on every call
mc_nocrn = MonteCarlo(n_paths=100000, n_steps=252, seed=None)  # fresh random numbers on every call

pricer_crn   = Pricer(model=model, method=mc_crn,   product=product)
pricer_nocrn = Pricer(model=model, method=mc_nocrn, product=product)

for name, p in [("seed=42  ", pricer_crn), ("seed=None", pricer_nocrn)]:
    print(name, "delta", fd.delta(p, S0, T, r), " gamma", fd.gamma(p, S0, T, r))

seed=42   delta (0.6376292708698735, 0.0018237485052532312)  gamma (0.017137423228837004, 0.003323394890511542)
seed=None delta (0.0655399126595624, 3.2859791501773805)  gamma (-712.1157290454639, 1142.888336875631)


In [9]:
import sys; sys.path.insert(0, '..')
import numpy as np
from premia_junior.models.black_scholes import BlackScholes
from premia_junior.methods.monte_carlo import MonteCarlo
from premia_junior.methods.analytical import Analytical
from premia_junior.products.european import EuropeanCall
from premia_junior.engine.pricer import Pricer

S0, T, r = 100, 1.0, 0.05
model   = BlackScholes(mu=0.05, sigma=0.2)
product = EuropeanCall(K=100)

mc = MonteCarlo(n_paths=100000, n_steps=252, seed=42)
result = Pricer(model=model, method=mc, product=product).price(S0, T, r)
print(result)                                            # short, no giant array
print(np.isclose(result.price, result.samples.mean()))   # True
print(result.samples.shape)                              # (100000,)

res_a = Pricer(model=model, method=Analytical(), product=product).price(S0, T, r)
print(res_a.samples)                                     # None

PricingResult(price=10.482858080322606, confidence_interval=(10.390945307384213, 10.574770853260997), metadata={'Method': 'MonteCarlo', 'Model': 'BlackScholes', 'Product': 'EuropeanCall'})
True
(100000,)
None


In [10]:
import premia_junior.engine.result as m
print(m.__file__)
print(m.PricingResult.__dataclass_fields__['samples'])

c:\Users\ASUS-PC\Desktop\premia_junior\notebooks\..\premia_junior\engine\result.py
Field(name='samples',type=numpy.ndarray | None,default=None,default_factory=<dataclasses._MISSING_TYPE object at 0x0000021B41502540>,init=True,repr=False,hash=None,compare=True,metadata=mappingproxy({}),kw_only=False,_field_type=_FIELD)


In [1]:
import sys; sys.path.insert(0, '..')
import numpy as np
from premia_junior.models.black_scholes import BlackScholes
from premia_junior.methods.analytical import Analytical
from premia_junior.methods.monte_carlo import MonteCarlo
from premia_junior.methods.CRR import CRR
from premia_junior.products.european import EuropeanCall
from premia_junior.engine.pricer import Pricer
from premia_junior.utils.greek import FiniteDifference
from premia_junior.utils.BaseGreekScheme import CentralDifference1

S0, T, r = 100, 1.0, 0.05
model, product = BlackScholes(mu=0.05, sigma=0.2), EuropeanCall(K=100)
fd = FiniteDifference(CentralDifference1(), h=0.01)

def show(name, res):
    val, se = res                      
    print(f"{name:6s} {val:12.5f}" + (f"  ± {1.96*se:.5f}" if se is not None else ""))

def run(label, method):
    print(label)
    pricer = Pricer(model=model, method=method, product=product)
    for g in ["delta", "gamma", "rho", "theta","vega"]:
        show(g, getattr(fd, g)(pricer, S0, T, r))

run("Analytical",          Analytical())
run("CRR (500 steps)",     CRR(n_steps=500))
run("MC seed=42",          MonteCarlo(n_paths=100000, n_steps=252, seed=42))
run("MC seed=None",        MonteCarlo(n_paths=100000, n_steps=252, seed=None))

Analytical
delta       0.63683
gamma       0.01876
rho        53.22477
theta      -6.41408
vega       37.52098
CRR (500 steps)
delta       0.63677
gamma       3.35457
rho        53.22241
theta      -6.41208
vega       37.50216
MC seed=42
delta       0.63763  ± 0.00357
gamma       0.01714  ± 0.00651
rho        53.26264  ± 0.28880
theta      -6.43764  ± 0.05547
vega       37.73539  ± 0.47493
MC seed=None
delta       2.94569  ± 6.44751
gamma    1138.11486  ± 2233.54175
rho        55.73252  ± 6.43773
theta      -8.77513  ± 6.46909
vega       30.26667  ± 6.44549
